# Advanced: Multi-Strategy Privacy Metrics with PAMOLA.CORE

**Goal**: Master all 3 privacy assessment strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Basic privacy metrics (DCR, NNDR, Uniqueness with defaults)
- **Strategy 2**: Distance-focused assessment (Advanced DCR with FAISS, custom aggregation)
- **Strategy 3**: Identity-focused assessment (Advanced k-anonymity, l-diversity, t-closeness)
- Compare re-identification risks across strategies
- Analyze privacy protection effectiveness
- Production deployment patterns for privacy assessment

**Prerequisites:**
- Completed the simple notebook
- Understanding of privacy concepts (k-anonymity, re-identification)
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── metrics/
        └── 02_privacy_metrics_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.metrics.operations.privacy_ops import PrivacyMetricOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record paired datasets
- Auto-creates sample if files not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 7 columns each)
- Sample records (first 5 rows from both datasets)
- Column statistics (unique values, ranges)

**Dataset features:**
- Original: Sensitive personal data (age, income, location, health)
- Transformed: Anonymized with noise, generalization, suppression
- Realistic privacy risks for testing all metrics

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
original_path = examples_dir / 'data_examples' / 'privacy_original_data.csv'
transformed_path = examples_dir / 'data_examples' / 'privacy_transformed_data.csv'

print(f"📂 Looking for data at:")
print(f"   Original: {original_path}")
print(f"   Transformed: {transformed_path}\n")

if original_path.exists() and transformed_path.exists():
    print(f"📂 Loading data from files")
    original_df = read_csv(original_path)
    transformed_df = read_csv(transformed_path)
    print(f"✓ Loaded {len(original_df)} records from each file")
else:
    print("📊 Generating synthetic datasets (1000 records each)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate original data with sensitive information
    original_df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'age': np.random.randint(18, 80, n_records),
        'income': np.random.randint(20000, 200000, n_records),
        'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_records),
        'city': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix', 
                                  'San Francisco', 'Seattle', 'Boston'], n_records),
        'zip_code': np.random.randint(10000, 99999, n_records),
        'health_score': np.random.uniform(50, 100, n_records).round(2)
    })
    
    # Generate transformed data with privacy-preserving techniques
    np.random.seed(43)
    transformed_df = original_df.copy()
    
    # Add noise to numeric fields
    transformed_df['age'] = (transformed_df['age'] + 
                            np.random.normal(0, 3, n_records)).astype(int).clip(18, 80)
    transformed_df['income'] = (transformed_df['income'] + 
                               np.random.normal(0, 8000, n_records)).astype(int).clip(20000, 200000)
    transformed_df['health_score'] = (transformed_df['health_score'] + 
                                     np.random.uniform(-8, 8, n_records)).clip(50, 100).round(2)
    
    # Generalize zip codes (last 2 digits to 00)
    transformed_df['zip_code'] = (transformed_df['zip_code'] // 100) * 100
    
    # Save for future use (with error handling)
    try:
        original_path.parent.mkdir(parents=True, exist_ok=True)
        original_df.to_csv(original_path, index=False)
        transformed_df.to_csv(transformed_path, index=False)
        print(f"✓ Generated and saved to:")
        print(f"   {original_path}")
        print(f"   {transformed_path}")
    except PermissionError:
        print(f"⚠️  Cannot save files (may be open)")
        print(f"   Datasets generated in memory only")

print(f"\n📊 ORIGINAL Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(original_df):,}")
print(f"  Columns: {', '.join(original_df.columns)}")

print(f"\n🔍 Sample Records (Original):")
print(original_df.head())

print(f"\n📊 TRANSFORMED Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(transformed_df):,}")
print(f"  Columns: {', '.join(transformed_df.columns)}")

print(f"\n🔍 Sample Records (Transformed):")
print(transformed_df.head())

print(f"\n📈 Column Statistics:")
for col in ['age', 'income', 'health_score', 'zip_code']:
    if col in original_df.columns and col in transformed_df.columns:
        orig_mean = original_df[col].mean()
        trans_mean = transformed_df[col].mean()
        orig_unique = original_df[col].nunique()
        trans_unique = transformed_df[col].nunique()
        
        print(f"  {col:15s}: Original mean={orig_mean:8.2f}, unique={orig_unique:4d}")
        print(f"  {' ':15s}  Transformed mean={trans_mean:8.2f}, unique={trans_unique:4d}")
        print()

print(f"💡 Perfect paired datasets for testing all 3 privacy strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit column names for each strategy
   - `numeric_columns`: For distance-based metrics (DCR, NNDR)
   - `quasi_identifiers`: For k-anonymity analysis
   - `sensitive_attributes`: For l-diversity, t-closeness
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field in both datasets)
- Column usage summary (distance vs identity metrics)
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource with 2 datasets created (✓)

**Note:** All columns must exist in both original and transformed datasets

**Field configuration:**
```python
FIELD_CONFIG = {
    "numeric_columns": ["age", "income", "health_score"],
    "quasi_identifiers": ["age", "zip_code"],
    "sensitive_attributes": ["health_score"]
}
```

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "numeric_columns": ["age", "income", "health_score"],  # For DCR/NNDR
    "quasi_identifiers": ["age", "zip_code"],  # For k-anonymity
    "sensitive_attributes": ["health_score"]  # For l-diversity, t-closeness
}

# Validate all fields exist in both datasets
print("📋 Validating field configuration...\n")

all_columns = list(dict.fromkeys(
    FIELD_CONFIG["numeric_columns"]
    + FIELD_CONFIG["quasi_identifiers"]
    + FIELD_CONFIG["sensitive_attributes"]
))
    

for col in all_columns:
    # Check original
    if col not in original_df.columns:
        raise ValueError(
            f"❌ Column '{col}' not found in original dataset!\n"
            f"Available columns: {', '.join(original_df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    # Check transformed
    if col not in transformed_df.columns:
        raise ValueError(
            f"❌ Column '{col}' not found in transformed dataset!\n"
            f"Available columns: {', '.join(transformed_df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    
    print(f"  ✓ '{col}':")
    print(f"      Original    - unique: {original_df[col].nunique():4d}")
    print(f"      Transformed - unique: {transformed_df[col].nunique():4d}")

print(f"\n📊 Field Usage:")
print(f"  Distance metrics (DCR/NNDR): {', '.join(FIELD_CONFIG['numeric_columns'])}")
print(f"  Identity metrics (k-anon):   {', '.join(FIELD_CONFIG['quasi_identifiers'])}")
print(f"  Sensitive attributes:        {', '.join(FIELD_CONFIG['sensitive_attributes'])}")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "original_dataset_name": "original_dataset",
        "transformed_dataset_name": "transformed_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_privacy_001",
    task_type="multi_strategy_privacy",
    description="Multi-strategy privacy metrics processing",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource with both datasets
data_source = DataSource(
    dataframes={
        "original_dataset": original_df,
        "transformed_dataset": transformed_df
    }
)
print("✓ DataSource created with 2 datasets")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Basic Privacy Metrics

**How to use:**
- Uses default parameters for all metrics
- Baseline assessment with standard settings
- Run to execute basic strategy

**Key parameters:**
- `privacy_metrics=['dcr', 'nndr', 'uniqueness']`: All three metric types
- `normalize=True`: Feature normalization
- `distance_metric='euclidean'`: Standard distance
- `k_values=[2, 3, 5]`: Basic k-anonymity thresholds

**What you'll see:**
- Configuration summary
- Progress bar: validation → calculation → metrics → viz → save
- Completion message with execution time (2-5 seconds)
- Privacy scores and risk assessments

**Validate:**
- ✅ Execution time <10 seconds
- ✅ All three metrics calculated
- ✅ Status = "completed"

**Note:** Best for quick baseline assessment

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: BASIC PRIVACY METRICS")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Basic metrics",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = PrivacyMetricOperation(
    # Core parameters
    name="privacy_basic",
    privacy_metrics=['dcr', 'nndr', 'uniqueness'],
    columns=all_columns,
    
    # Metric-specific parameters
    metric_params={
        'dcr': {
            'distance_metric': 'euclidean',
            'normalize_features': True,
            'percentiles': [5, 25, 50, 75, 95]
        },
        'nndr': {
            'distance_metric': 'euclidean',
            'normalize_features': True,
            'threshold': 0.5
        },
        'uniqueness': {
            'quasi_identifiers': FIELD_CONFIG["quasi_identifiers"],
            'sensitives': FIELD_CONFIG["sensitive_attributes"],
            'k_values': [2, 3, 5],
            'l_diversity': True,
            't_closeness': True
        }
    },
    
    # Processing settings
    normalize=True,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Metrics: {', '.join(operation_s1.privacy_metrics)}")
print(f"  Columns: {', '.join(operation_s1.columns)}")
print(f"  Normalize: {operation_s1.normalize}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_basic',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")
print(f"   Status: {result_s1.status}")
print(f"   Artifacts: {len(result_s1.artifacts)}")

# Load metrics
metrics_files_s1 = sorted(
    list((task_dir / 'strategy1_basic' / 'metrics').glob('*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if metrics_files_s1:
    # Filter out data_types files
    filtered_s1 = [f for f in metrics_files_s1 if not f.name.startswith("data_types")]
    if filtered_s1:
        with open(filtered_s1[0], 'r') as f:
            metrics_s1 = json.load(f).get('metrics', {})
            if 'dcr' in metrics_s1:
                privacy_score = metrics_s1['dcr'].get('privacy_score', 'N/A')
                print(f"\n📊 DCR Privacy Score: {privacy_score:.4f}" if isinstance(privacy_score, float) else f"\n📊 DCR Privacy Score: {privacy_score}")
            if 'uniqueness' in metrics_s1 and 'k_anonymity' in metrics_s1['uniqueness']:
                k_anon = metrics_s1['uniqueness']['k_anonymity']
                print(f"📊 Identity Disclosure: {k_anon.get('identity_disclosure_rate', 'N/A'):.2%}")

## STRATEGY 2: Distance-Focused Privacy Assessment

**How to use:**
- Advanced DCR with FAISS acceleration
- Custom aggregation (mean_k)
- Multiple distance metrics comparison
- Run to execute distance-focused strategy

**Key parameters:**
- `use_faiss=True`: FAISS acceleration for large datasets
- `aggregation='mean_k'`: Average of k nearest distances
- `k_neighbors=5`: Consider 5 nearest neighbors
- `percentiles=[1, 5, 10, 25, 50, 75, 90, 95, 99]`: Detailed distribution

**What you'll see:**
- Configuration confirmation
- Progress bar: validation → FAISS indexing → calculation → metrics → viz → save
- Completion message with execution time (1-3 seconds with FAISS)
- Advanced DCR and NNDR statistics

**Validate:**
- ✅ Execution time <5 seconds
- ✅ FAISS acceleration used
- ✅ Status = "completed"

**Note:** Best for large datasets requiring detailed distance analysis

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: DISTANCE-FOCUSED PRIVACY ASSESSMENT")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Distance-focused",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = PrivacyMetricOperation(
    name="privacy_distance",
    privacy_metrics=['dcr', 'nndr'],  # Focus on distance metrics only
    columns=all_columns,
    
    metric_params={
        'dcr': {
            'distance_metric': 'euclidean',
            'normalize_features': True,
            'use_faiss': True,  # Enable FAISS acceleration
            'aggregation': 'mean_k',  # Average of k nearest
            'k_neighbors': 5,
            'percentiles': [1, 5, 10, 25, 50, 75, 90, 95, 99]
        },
        'nndr': {
            'distance_metric': 'euclidean',
            'n_neighbors': 2,
            'normalize_features': True,
            'threshold': 0.4,  # Stricter threshold
            'realistic_threshold': 0.8,
            'at_risk_threshold': 0.4
        }
    },
    
    normalize=True,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  DCR: FAISS enabled, mean_k aggregation (k=5)")
print(f"  NNDR: Stricter threshold (0.4)")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_distance',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")
print(f"   Status: {result_s2.status}")
print(f"   Artifacts: {len(result_s2.artifacts)}")

# Load metrics
metrics_files_s2 = sorted(
    list((task_dir / 'strategy2_distance' / 'metrics').glob('*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if metrics_files_s2:
    filtered_s2 = [f for f in metrics_files_s2 if not f.name.startswith("data_types")]
    if filtered_s2:
        with open(filtered_s2[0], 'r') as f:
            metrics_s2 = json.load(f).get('metrics', {})
            if 'dcr' in metrics_s2:
                privacy_score = metrics_s2['dcr'].get('privacy_score', 'N/A')
                print(f"\n📊 DCR Privacy Score: {privacy_score:.4f}" if isinstance(privacy_score, float) else f"\n📊 DCR Privacy Score: {privacy_score}")
            if 'nndr' in metrics_s2:
                high_risk = metrics_s2['nndr'].get('high_risk_records', 'N/A')
                print(f"📊 NNDR High Risk Records: {high_risk}")

## STRATEGY 3: Identity-Focused Privacy Assessment

**How to use:**
- Advanced uniqueness metrics only
- Multiple k-anonymity thresholds
- Full l-diversity and t-closeness analysis
- Run to execute identity-focused strategy

**Key parameters:**
- `k_values=[1, 2, 3, 5, 10, 20]`: Comprehensive k-anonymity analysis
- `l_diversity=True`: Sensitive attribute diversity
- `t_closeness=True`: Distribution similarity within groups

**What you'll see:**
- Configuration confirmation
- Progress bar: validation → grouping → k-anon → l-div → t-close → viz → save
- Completion message with execution time (2-4 seconds)
- Detailed identity disclosure statistics

**Validate:**
- ✅ Execution time <8 seconds
- ✅ All k-values analyzed
- ✅ Status = "completed"

**Note:** Best for regulatory compliance (GDPR, HIPAA) requiring formal privacy guarantees

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: IDENTITY-FOCUSED PRIVACY ASSESSMENT")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Identity-focused",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = PrivacyMetricOperation(
    name="privacy_identity",
    privacy_metrics=['uniqueness'],  # Focus on identity metrics only
    columns=all_columns,  # Still need for compatibility
    
    metric_params={
        'uniqueness': {
            'quasi_identifiers': FIELD_CONFIG["quasi_identifiers"],
            'sensitives': FIELD_CONFIG["sensitive_attributes"],
            'k_values': [1, 2, 3, 5, 10, 20],  # Comprehensive analysis
            'l_diversity': True,
            't_closeness': True
        }
    },
    
    normalize=False,  # Not needed for identity metrics
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  K-anonymity: {operation_s3.metric_params['uniqueness']['k_values']}")
print(f"  L-diversity: {operation_s3.metric_params['uniqueness']['l_diversity']}")
print(f"  T-closeness: {operation_s3.metric_params['uniqueness']['t_closeness']}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_identity',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")
print(f"   Status: {result_s3.status}")
print(f"   Artifacts: {len(result_s3.artifacts)}")

# Load metrics
metrics_files_s3 = sorted(
    list((task_dir / 'strategy3_identity' / 'metrics').glob('*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if metrics_files_s3:
    filtered_s3 = [f for f in metrics_files_s3 if not f.name.startswith("data_types")]
    if filtered_s3:
        with open(filtered_s3[0], 'r') as f:
            metrics_s3 = json.load(f).get('metrics', {})
            if 'uniqueness' in metrics_s3 and 'k_anonymity' in metrics_s3['uniqueness']:
                k_anon = metrics_s3['uniqueness']['k_anonymity']
                print(f"\n📊 Identity Disclosure: {k_anon.get('identity_disclosure_rate', 'N/A'):.2%}")
                print(f"📊 Number of Groups: {k_anon.get('num_groups', 'N/A')}")

## Step 4: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and effectiveness

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Privacy score comparison (DCR)
- Identity disclosure comparison (k-anonymity)
- Risk assessment summary

**Comparison metrics:**
- Higher privacy score = Better protection
- Lower identity disclosure = Better protection
- Lower high-risk records (NNDR) = Better protection

**Note:** Different strategies focus on different aspects - comprehensive assessment requires all three

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Basic):           {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Distance-focused): {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Identity-focused): {elapsed_s3:6.2f}s")
print(f"  Total:                         {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

# Load all metrics for comparison
if 'metrics_s1' in locals() and 'metrics_s2' in locals() and 'metrics_s3' in locals():
    print("\n📈 Privacy Score Comparison (DCR):")
    if 'dcr' in metrics_s1:
        score_s1 = metrics_s1['dcr'].get('privacy_score', 'N/A')
        print(f"  Strategy 1: {score_s1:.4f}" if isinstance(score_s1, float) else f"  Strategy 1: {score_s1}")
    if 'dcr' in metrics_s2:
        score_s2 = metrics_s2['dcr'].get('privacy_score', 'N/A')
        print(f"  Strategy 2: {score_s2:.4f}" if isinstance(score_s2, float) else f"  Strategy 2: {score_s2}")
    
    print("\n📉 Identity Disclosure Comparison (k-anonymity):")
    if 'uniqueness' in metrics_s1 and 'k_anonymity' in metrics_s1['uniqueness']:
        id_rate_s1 = metrics_s1['uniqueness']['k_anonymity'].get('identity_disclosure_rate', 0)
        print(f"  Strategy 1: {id_rate_s1:.2%}")
    if 'uniqueness' in metrics_s3 and 'k_anonymity' in metrics_s3['uniqueness']:
        id_rate_s3 = metrics_s3['uniqueness']['k_anonymity'].get('identity_disclosure_rate', 0)
        print(f"  Strategy 3: {id_rate_s3:.2%}")
    
    print("\n🔒 High Risk Assessment (NNDR):")
    if 'nndr' in metrics_s1:
        hr_s1 = metrics_s1['nndr'].get('high_risk_records', 0)
        print(f"  Strategy 1: {hr_s1} records")
    if 'nndr' in metrics_s2:
        hr_s2 = metrics_s2['nndr'].get('high_risk_records', 0)
        print(f"  Strategy 2: {hr_s2} records")
    
    print("\n💡 Interpretation Guide:")
    print("  - Higher privacy score (DCR) = Better distance-based protection")
    print("  - Lower identity disclosure = Better k-anonymity protection")
    print("  - Fewer high-risk records (NNDR) = Better neighbor-based protection")
    print("  - Strategy 2 uses FAISS for faster computation on large datasets")
else:
    print("\n⚠️  Run all strategies first to see comparison!")

## Step 5: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: DCR statistics, NNDR ratios, k-anonymity violations, l-diversity, t-closeness
2. **Visualizations**: Privacy risk distributions (latest batch only)

**Validate:**
- ✅ All privacy scores present
- ✅ Risk assessments complete
- ✅ K-anonymity violations calculated
- ✅ Visualizations generated

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_basic', 'Strategy 1: Basic Privacy Metrics'),
    ('strategy2_distance', 'Strategy 2: Distance-Focused Assessment'),
    ('strategy3_identity', 'Strategy 3: Identity-Focused Assessment')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # DCR
                if 'dcr' in metrics:
                    dcr = metrics['dcr']
                    print(f"\n   📈 DCR:")
                    if 'dcr_statistics' in dcr:
                        stats = dcr['dcr_statistics']
                        print(f"      Mean: {stats.get('mean', 'N/A'):.4f}")
                        print(f"      Min: {stats.get('min', 'N/A'):.4f}")
                    print(f"      Privacy Score: {dcr.get('privacy_score', 'N/A'):.4f}")
                    
                # NNDR
                if 'nndr' in metrics:
                    nndr = metrics['nndr']
                    print(f"\n   📈 NNDR:")
                    if 'nndr_statistics' in nndr:
                        stats = nndr['nndr_statistics']
                        print(f"      Mean: {stats.get('mean', 'N/A'):.4f}")
                    print(f"      High Risk Records: {nndr.get('high_risk_records', 'N/A')}")
                    
                # Uniqueness
                if 'uniqueness' in metrics:
                    uniq = metrics['uniqueness']
                    print(f"\n   📈 Uniqueness:")
                    if 'k_anonymity' in uniq:
                        k_anon = uniq['k_anonymity']
                        print(f"      Identity Disclosure: {k_anon.get('identity_disclosure_rate', 'N/A'):.2%}")
                        print(f"      Number of Groups: {k_anon.get('num_groups', 'N/A')}")
                        if 'k_anonymity_stats' in k_anon and k_anon['k_anonymity_stats']:
                            print(f"      K-anonymity violations:")
                            for stat in k_anon['k_anonymity_stats'][:3]:  # Show first 3
                                k_val = stat['k_value']
                                violations = stat['num_violations']
                                pct = stat['percent_violation']
                                print(f"        k={k_val}: {violations} violations ({pct:.1f}%)")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(list(viz_dir.glob('*.png')),
                          key=lambda x: x.stat().st_mtime, reverse=True)
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Privacy Assessment Summary

**How to use:**
- Run to get overall privacy assessment
- Compare all strategies holistically

**What you'll see:**
- Best strategy recommendation per risk type
- Overall privacy rating
- Key insights from each strategy
- Re-identification risk summary

**Assessment criteria:**
- ✅ High privacy: Privacy score > 0.7, identity disclosure < 5%, few high-risk records
- ⚠️  Medium privacy: Privacy score 0.4-0.7, identity disclosure 5-20%
- ❌ Low privacy: Privacy score < 0.4, identity disclosure > 20%

**Note:** Consider use case when selecting strategy - regulatory compliance often requires Strategy 3

In [ ]:
print("\n" + "=" * 80)
print("🎯 PRIVACY ASSESSMENT SUMMARY")
print("=" * 80 + "\n")

if 'metrics_s1' in locals() and 'metrics_s2' in locals() and 'metrics_s3' in locals():
    # Collect all privacy indicators
    privacy_indicators = {}
    
    # Strategy 1
    if 'dcr' in metrics_s1:
        privacy_indicators['dcr_s1'] = metrics_s1['dcr'].get('privacy_score', 0)
    if 'uniqueness' in metrics_s1:
        privacy_indicators['disclosure_s1'] = metrics_s1['uniqueness']['k_anonymity'].get('identity_disclosure_rate', 1)
    
    # Strategy 2
    if 'dcr' in metrics_s2:
        privacy_indicators['dcr_s2'] = metrics_s2['dcr'].get('privacy_score', 0)
    if 'nndr' in metrics_s2:
        privacy_indicators['high_risk_s2'] = metrics_s2['nndr'].get('high_risk_records', float('inf'))
    
    # Strategy 3
    if 'uniqueness' in metrics_s3:
        privacy_indicators['disclosure_s3'] = metrics_s3['uniqueness']['k_anonymity'].get('identity_disclosure_rate', 1)
    
    print("📊 Privacy Assessment by Strategy:\n")
    
    # Strategy 1
    print("  Strategy 1 (Basic):")
    dcr_score = privacy_indicators.get('dcr_s1', 0)
    disclosure = privacy_indicators.get('disclosure_s1', 1)
    print(f"    DCR Privacy Score: {dcr_score:.4f}")
    print(f"    Identity Disclosure: {disclosure:.2%}")
    
    if dcr_score > 0.7 and disclosure < 0.05:
        print(f"    Assessment: ✅ HIGH PRIVACY")
    elif dcr_score > 0.4 and disclosure < 0.20:
        print(f"    Assessment: ⚠️  MEDIUM PRIVACY")
    else:
        print(f"    Assessment: ❌ LOW PRIVACY")
    print()
    
    # Strategy 2
    print("  Strategy 2 (Distance-focused):")
    dcr_score_s2 = privacy_indicators.get('dcr_s2', 0)
    high_risk = privacy_indicators.get('high_risk_s2', 0)
    print(f"    DCR Privacy Score: {dcr_score_s2:.4f}")
    print(f"    High Risk Records: {high_risk}")
    
    if dcr_score_s2 > 0.7 and high_risk < len(original_df) * 0.05:
        print(f"    Assessment: ✅ HIGH PRIVACY")
    elif dcr_score_s2 > 0.4:
        print(f"    Assessment: ⚠️  MEDIUM PRIVACY")
    else:
        print(f"    Assessment: ❌ LOW PRIVACY")
    print()
    
    # Strategy 3
    print("  Strategy 3 (Identity-focused):")
    disclosure_s3 = privacy_indicators.get('disclosure_s3', 1)
    print(f"    Identity Disclosure: {disclosure_s3:.2%}")
    
    if disclosure_s3 < 0.05:
        print(f"    Assessment: ✅ HIGH PRIVACY")
    elif disclosure_s3 < 0.20:
        print(f"    Assessment: ⚠️  MEDIUM PRIVACY")
    else:
        print(f"    Assessment: ❌ LOW PRIVACY")
    
    print("\n" + "=" * 80)
    print("\n💡 Key Insights:")
    print("  - Strategy 1 provides balanced view of all privacy aspects")
    print("  - Strategy 2 excels at distance-based re-identification detection")
    print("  - Strategy 3 provides formal privacy guarantees (k-anonymity)")
    
    print("\n📋 Use Case Recommendations:")
    print("  • Strategy 1: General-purpose privacy assessment")
    print("  • Strategy 2: Large datasets requiring FAISS acceleration")
    print("  • Strategy 3: Regulatory compliance (GDPR, HIPAA)")
    
else:
    print("⚠️  Run all strategies first to see assessment!")

## 🎯 Summary

**Accomplished:**
- ✅ 3 privacy strategies implemented and compared
- ✅ Distance-based metrics (DCR, NNDR) calculated
- ✅ Identity-based metrics (k-anonymity, l-diversity, t-closeness) calculated
- ✅ FAISS acceleration tested for large datasets
- ✅ Production-ready artifacts generated

**Key takeaways:**
- **Distance metrics**: DCR and NNDR measure proximity to original records
- **Identity metrics**: K-anonymity measures group-based protection
- **Multiple perspectives**: Comprehensive assessment requires all metrics
- **FAISS acceleration**: Dramatically faster for datasets >10K records

**Strategy recommendations:**
- **Use Strategy 1** when: Quick baseline assessment needed, mixed data types
- **Use Strategy 2** when: Large datasets (>10K records), distance-based risks primary concern
- **Use Strategy 3** when: Regulatory compliance required, formal privacy guarantees needed

**Privacy improvement strategies:**
- Low privacy score: Add more noise, increase suppression
- High identity disclosure: Generalize quasi-identifiers further
- Many high-risk records: Review and refine anonymization parameters

**Next steps:**
- Test with your own datasets
- Tune parameters based on privacy requirements
- Integrate into anonymization pipelines
- Monitor privacy metrics in production

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)